# Day 21 — Week 3 Finale: Agent × Graph × LLM Research Map

**Goal: no major new topic.** Consolidate Week 3 and prepare for Week 4: **Paper → Repo → Baseline → Experiment**.

```text
LLM Inference → Agent / ReAct → Mini Agent → Graph × LLM → Graph RAG / KG → Research Question
```


## 0. Week 3 in one picture

```text
                         LLM
                    _____|_____
                   /           \
          Inference/Serving    Agent / ReAct
                               |
                         Thought → Action
                            ↑       ↓
                         Observation
                               |
                          Graph × LLM
                         /           \
                LLM for Graph    Graph for LLM
                                      |
                              Graph RAG / Graph Agent
```

The goal is not memorizing boxes. Understand **where information comes from, how it is represented, how the model interacts with it, and where failure occurs**.

### Mandatory Question 1
Explain **LLM for Graph** vs **Graph for LLM**, with one example of each.

Answer:

- **LLM for Graph:** use the semantic and reasoning capabilities of LLMs to improve graph-related tasks, such as node classification, link prediction, or graph generation.

- **Graph for LLM:** use explicit graph structure and relational knowledge to improve LLM retrieval, reasoning, memory, or agent behavior. For example, Graph RAG retrieves relevant entities, relations, and subgraphs to provide structured evidence for an LLM.

## 1. From ordinary LLM to Agent

```text
ordinary: Prompt → LLM → Answer

agent:
Task → LLM Decision → Action/Tool → Environment
 ↑                                  ↓
 └──────── state/trajectory ← Observation
```

ReAct gives the conceptual loop:

```text
Thought → Action → Observation → Thought → ...
```

The key change is **external feedback**.

### Mandatory Question 2
Why is an Agent more than “an LLM that can call a function”? Explain decision, action/tool, observation, state/trajectory, and termination.

Answer: An Agent is more than an LLM that can call a function because it maintains a decision loop over time. The LLM makes a decision, selects an action or tool, receives an Observation from the environment, updates its state/trajectory, and then uses that updated information to decide what to do next. The process continues until a termination condition is reached.

## 2. Why Graph enters the Agent story

A graph can serve as structured knowledge, retrievable evidence, memory, an interactive environment, or a constrained action space.

```text
Question → Agent decision → graph_lookup(entity, relation)
         → Graph Observation → state update → next decision
```

### Mandatory Question 3
Why can `graph_lookup(entity, relation)` be viewed as a tiny Graph-Agent interface, although it is not itself a full Agent?

Answer: graph_lookup(entity, relation) can be viewed as a tiny Graph-Agent interface because it allows an Agent to perform an action on a graph and receive graph information as an Observation. However, it is not itself a full Agent because it does not make decisions, maintain state/trajectory, choose the next action, or determine when to terminate.

## 3. Graph RAG pipeline

```text
Documents / Existing KG
        ↓
Entity & Relation Representation
        ↓
Entity Linking
        ↓
Graph Retrieval
        ↓
Relevant Subgraph
        ↓
Serialization / Context Construction
        ↓
LLM Reasoning
        ↓
Answer
```

Every arrow can become a research bottleneck.

### Mandatory Question 4
Give **three different failure stages** that could independently cause a wrong final answer. Why does final answer accuracy alone not identify which component failed?

Answer: Three possible failure stages are entity linking, graph retrieval, and graph reasoning.
Entity linking may map the query to the wrong entity, graph retrieval may fail to retrieve the necessary subgraph, and graph reasoning may incorrectly interpret the retrieved relations. Final answer accuracy alone cannot tell us which component failed because all three different errors may produce the same wrong final answer.

## 4. Microsoft GraphRAG — connect the real system to our toy model

The overview you just read adds an important idea: **indexing is not merely storing triples**.

```text
Input Corpus
   ↓
TextUnits
   ↓
Entity / Relationship / Claim Extraction
   ↓
Graph
   ↓
Hierarchical Community Detection
   ↓
Community Summaries
```

At query time:

- **Local Search**: questions around specific entities; fan out through nearby entities/relations.
- **Global Search**: holistic questions about the corpus; leverage community summaries.
- **DRIFT Search**: starts locally but incorporates broader community information.
- **Basic Search**: baseline RAG using standard top-k vector search.

The important idea is **retrieval scope**: different questions need different evidence.

### Mandatory Question 5
Which signal is most natural and why?

A. “Where was the conference that published ReAct held?”  
B. “What are the major themes across this entire collection of AI-agent papers?”  
C. “Find passages semantically similar to ‘tool-use learning’.”

Choose mainly from Local / Global / Basic-vector. The labels are not absolute.

- Local:A
- Global:B
- Basic:C

## 5. Vector RAG, Graph RAG, Hybrid RAG

```text
Vector RAG:
Question → embedding → semantic similarity → chunks → LLM

Graph RAG:
Question → entity/relation grounding → graph/subgraph → structured evidence → LLM

Hybrid:
Question ─┬→ vector evidence ─┐
          └→ graph evidence  ─┴→ fusion → LLM
```

Graph retrieval is not universally better. Hybrid retrieval can also add noise, contradiction, latency, and context cost.

### Mandatory Question 6 — Ablation
Design an experiment testing whether the **graph component actually contributes**, rather than merely giving the LLM more context.

State: baseline, intervention, controlled variables, and metrics.

Answer:  
- Baseline: Vector RAG only.
- Intervention: Add graph-structured retrieval while keeping the total evidence/context budget approximately fixed.
- Controlled variables: Same dataset, questions, LLM, prompts, decoding settings, and context budget.
- Metrics: Final answer accuracy, retrieval recall/precision, latency, and token cost.

## 6. Week 2 still matters

GCN, GraphSAGE, and GAT are members of the broader **GNN** family.

A possible hybrid system is:

```text
node text → LLM encoder → node embeddings → GNN/graph encoder
                                      ↓
                           retrieval / prediction
                                      ↓
                                LLM / Agent
```

But Graph × LLM does **not automatically require a GNN**.

### Mandatory Question 7
Respond carefully:

> “If I already have a powerful LLM and a graph database, GNNs are unnecessary.”

When might a GNN help, and when might it be unnecessary?

Answer:A GNN is not always necessary when an LLM can directly use a graph database for explicit relation lookup, path traversal, or subgraph retrieval. However, a GNN can be useful when the system needs learned graph representations, neighborhood aggregation, node or edge prediction, or structural generalization beyond explicit database queries. Therefore, whether a GNN is useful depends on whether the task requires graph learning or only graph access.

## 7. Day 16 still matters too

Graph Agents can generate dynamic serving workloads:

```text
Agent step 1 → LLM → graph retrieval
Agent step 2 → LLM → tool
Agent step 3 → LLM → graph retrieval
...
```

Step count, retrieved context size, tool latency, and termination time can all vary. So latency/throughput, KV cache, batching/scheduling, and memory traffic remain relevant — but this is a **supporting systems lens**, not the center of your current direction.

### Mandatory Question 8
Why can a Graph Agent create a more dynamic workload than single-turn RAG? Connect at least three: variable steps, tool latency, context growth, subgraph size, termination, batching.

Answer: A Graph Agent creates a more dynamic workload because the number of Agent steps is not fixed, each step may involve tools or graph retrieval with different latency, and the retrieved observations can have different sizes. As the trajectory grows, the LLM context also grows, which increases token cost and KV-cache usage. Different requests may terminate at different times, so batching and scheduling become more difficult than in single-turn RAG.

## 8. Research decomposition

```text
Task → Pipeline → Bottleneck → Hypothesis → Intervention
     → Controlled Experiment / Ablation → Metrics → Conclusion
```

Weak question:

> Can I combine Graph + LLM + Agent?

Better question:

> On multi-hop QA, does noisy subgraph retrieval cause reasoning failures, and can relation-aware pruning improve evidence precision without reducing answer coverage?

### Mandatory Question 9 — Build one research question
Choose one bottleneck: entity linking, graph construction, subgraph retrieval, serialization, multi-hop reasoning, evidence fusion, graph memory, graph tool selection, etc.

Write:
1. Task
2. Bottleneck
3. Why current systems may fail
4. Hypothesis
5. Intervention
6. Ablation / controlled comparison
7. Metrics

Answer:

1. Task:
Multi-hop question answering with Graph RAG

2. Bottleneck:
The retrieval depth (number of graph hops) is difficult to choose.
Too few hops may miss necessary evidence, while too many hops may introduce irrelevant nodes and noise.

3. Why current systems may fail:
A fixed hop number may not fit all questions.
Some questions require only 1–2 hops, while others require deeper relational evidence.

4. Hypothesis:
An adaptive hop-selection strategy can improve answer accuracy and retrieval efficiency compared with a fixed-hop retriever.

5. Intervention:
Design a query-aware adaptive retrieval method that dynamically decides when to stop graph expansion.

6. Ablation / controlled comparison:
Compare:
- fixed 1-hop
- fixed 2-hop
- fixed 3-hop
- adaptive-hop retrieval

Keep the dataset, LLM, prompt, and context budget controlled as much as possible.

7. Metrics:
- answer accuracy
- evidence recall
- retrieval precision
- subgraph size
- latency / token cost

## 9. Your current research map

```text
                 Structured Reasoning for LLMs
                          |
              ____________|____________
             |            |            |
        Graph × LLM     Agents     Post-training
             |
       ______|_______
      |      |       |
 Graph RAG   KG   Graph Agent
      |
retrieval → structured evidence → multi-hop reasoning
```

Supporting capability:

```text
Transformer / CS336 / LLM inference systems
```

### Mandatory Question 10 — Your boundary
In 3–5 sentences explain your current research direction **without merely saying “Graph + LLM.”** Include the problem, why graph structure may help, where Agent/structured reasoning fits, and what still requires evidence.

Answer:
My current research interest is structured reasoning for LLMs, especially problems that require relational and multi-hop reasoning. Explicit graph structure may help by organizing entities, relations, and evidence into structured paths or subgraphs instead of relying only on unstructured text. Agents can further interact with the graph dynamically by retrieving evidence, choosing relations or paths, and deciding when enough information has been collected. However, whether these methods truly improve reasoning accuracy, efficiency, and reliability still needs to be validated through controlled experiments and ablations.
 

# Week 3 Final Recap

Try without looking back:

1. Why is autoregressive decoding sequential?
2. Prefill vs Decode?
3. Why does KV cache help?
4. What does PagedAttention mainly solve?
5. What is the ReAct loop?
6. Agent vs one-shot LLM?
7. What is Observation and why must it update state?
8. LLM for Graph?
9. Graph for LLM?
10. Vector RAG vs Graph RAG?
11. Entity linking and subgraph retrieval?
12. Why can retrieval succeed while reasoning fails?
13. Why does graph serialization matter?
14. What is a Graph Agent?
15. Where can GNNs still fit?
16. Why is more retrieval not always better?
17. What is an ablation for?
18. Name one Graph × LLM bottleneck you would investigate.

> **Week 3 target:** Graph-structured information can serve as model input, retrievable evidence, memory, or an interactive environment; LLMs and Agents can use this structure for retrieval and reasoning, while each stage creates distinct modeling, systems, and evaluation bottlenecks.


# Transition to Week 4 — Research Training

**Stop expanding the taxonomy. Start touching real research.**

```text
Choose one concrete problem
        ↓
Read one representative paper deeply
        ↓
Understand its repository
        ↓
Reproduce one baseline/result
        ↓
Identify one bottleneck
        ↓
Make one small controlled modification
        ↓
Experiment + ablation
        ↓
Explain the evidence
```

Checklist:

```text
[ ] Paper Reading
[ ] Repository Understanding
[ ] Baseline Reproduction
[ ] Small Modification Experiment
```

You do **not** need a publishable idea immediately. First become able to move from **paper claim → code → experiment → evidence**.


# Final Deliverable

Create `week3_agent_graph_llm_summary.md` with:

1. **What I learned** — compact Day 16–21 summary.
2. **My research map** — your own LLM → Agent → Graph × LLM map.
3. **Three bottlenecks I find interesting** — rank them.
4. **One candidate Week 4 direction** — allowed to change after paper reading.
5. **What I still do not understand** — keep precise uncertainties.
